In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import wfdb
from datetime import datetime, timedelta
from datasets import load_dataset
import matplotlib.pyplot as plt
import math
import sys
import os

sys.path.append("../src")
from llms import MyOpenAIModel, image_to_base64
import PIL.Image

/home/helenjin/miniconda3/envs/ares/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("BrachioLab/mcmed-cardiac")
train_data = ds['train']
train_data

Dataset({
    features: ['label', 'alarm', 'record_name', 'n_sig', 'fs', 'counter_freq', 'base_counter', 'sig_len', 'base_time', 'base_date', 'comments', 'sig_name', 'p_signal', 'd_signal', 'e_p_signal', 'e_d_signal', 'file_name', 'fmt', 'samps_per_frame', 'skew', 'byte_offset', 'adc_gain', 'baseline', 'units', 'adc_res', 'adc_zero', 'init_value', 'checksum', 'block_size', 'MRN', 'CSN', 'Visit_no', 'Visits', 'Age', 'Gender', 'Race', 'Ethnicity', 'Means_of_arrival', 'Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP', 'Triage_acuity', 'CC', 'ED_dispo', 'Hours_to_next_visit', 'Dispo_class_next_visit', 'ED_LOS', 'Hosp_LOS', 'DC_dispo', 'Payor_class', 'Admit_service', 'Dx_ICD9', 'Dx_ICD10', 'Dx_name', 'Arrival_time', 'Roomed_time', 'Dispo_time', 'Admit_time', 'Departure_time'],
    num_rows: 317
})

In [6]:
annotation_examples = ['99582725_1', '99441004_1', '99060008_1', '99907876_1', '99849825_1']

In [9]:
filtered_data = train_data.filter(lambda x: x["record_name"] in annotation_examples)

print(filtered_data)

Filter:   0%|          | 0/317 [00:00<?, ? examples/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7abc4da258a0>>
Traceback (most recent call last):
  File "/home/helenjin/miniconda3/envs/ssg-env/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


Dataset({
    features: ['label', 'alarm', 'record_name', 'n_sig', 'fs', 'counter_freq', 'base_counter', 'sig_len', 'base_time', 'base_date', 'comments', 'sig_name', 'p_signal', 'd_signal', 'e_p_signal', 'e_d_signal', 'file_name', 'fmt', 'samps_per_frame', 'skew', 'byte_offset', 'adc_gain', 'baseline', 'units', 'adc_res', 'adc_zero', 'init_value', 'checksum', 'block_size', 'MRN', 'CSN', 'Visit_no', 'Visits', 'Age', 'Gender', 'Race', 'Ethnicity', 'Means_of_arrival', 'Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP', 'Triage_acuity', 'CC', 'ED_dispo', 'Hours_to_next_visit', 'Dispo_class_next_visit', 'ED_LOS', 'Hosp_LOS', 'DC_dispo', 'Payor_class', 'Admit_service', 'Dx_ICD9', 'Dx_ICD10', 'Dx_name', 'Arrival_time', 'Roomed_time', 'Dispo_time', 'Admit_time', 'Departure_time'],
    num_rows: 5
})


In [18]:
for example in filtered_data: 
    print(example['label'], example['alarm'])
    
    record_keys = ['record_name', 'n_sig', 'fs', 'counter_freq', 'base_counter', 'sig_len', 'base_time', 'base_date', 'comments', 'sig_name', 'p_signal', 'd_signal', 'e_p_signal', 'e_d_signal', 'file_name', 'fmt', 'samps_per_frame', 'skew', 'byte_offset', 'adc_gain', 'baseline', 'units', 'adc_res', 'adc_zero', 'init_value', 'checksum', 'block_size']
    record_dict = {k: v for k, v in example.items() if k in record_keys}
    record_dict['p_signal'] = np.array(record_dict['p_signal']) 
    record = wfdb.Record(**record_dict)

    wfdb.plot_wfdb(record, return_fig=True)
    record_name = record.__dict__['record_name']
    plt.savefig(f'_dump/cardiac/annotation_graphs/{record_name}.png', dpi=200, bbox_inches='tight')
    plt.close()
    
    # wfdb.plot_wfdb(record=record)

False None
False None
True 2295-05-09 19:06:55
True 2271-07-22 07:53:28
True 2206-07-02 21:48:04


## add the relevance filtering examples

In [ ]:
# cardiac_relevance_fewshot_1_image_99593648_1.png
# cardiac_relevance_fewshot_2_image_99877003_1.png
# cardiac_relevance_fewshot_3_image_99579278_1.png
# cardiac_relevance_fewshot_4_image_99318009_1.png
# cardiac_relevance_fewshot_5_image_99972446_1.png

In [1]:
annotation_examples = ['99593648_1', '99877003_1', '99579278_1', '99318009_1', '99972446_1']


In [4]:
filtered_data = train_data.filter(lambda x: x["record_name"] in annotation_examples)

print(filtered_data)

Filter: 100%|███████████████████████████████████████████████████████████████████████| 317/317 [00:24<00:00, 12.79 examples/s]

Dataset({
    features: ['label', 'alarm', 'record_name', 'n_sig', 'fs', 'counter_freq', 'base_counter', 'sig_len', 'base_time', 'base_date', 'comments', 'sig_name', 'p_signal', 'd_signal', 'e_p_signal', 'e_d_signal', 'file_name', 'fmt', 'samps_per_frame', 'skew', 'byte_offset', 'adc_gain', 'baseline', 'units', 'adc_res', 'adc_zero', 'init_value', 'checksum', 'block_size', 'MRN', 'CSN', 'Visit_no', 'Visits', 'Age', 'Gender', 'Race', 'Ethnicity', 'Means_of_arrival', 'Triage_Temp', 'Triage_HR', 'Triage_RR', 'Triage_SpO2', 'Triage_SBP', 'Triage_DBP', 'Triage_acuity', 'CC', 'ED_dispo', 'Hours_to_next_visit', 'Dispo_class_next_visit', 'ED_LOS', 'Hosp_LOS', 'DC_dispo', 'Payor_class', 'Admit_service', 'Dx_ICD9', 'Dx_ICD10', 'Dx_name', 'Arrival_time', 'Roomed_time', 'Dispo_time', 'Admit_time', 'Departure_time'],
    num_rows: 5
})


In [12]:
for i, example in enumerate(filtered_data): 
    print(example['label'], example['alarm'])
    
    record_keys = ['record_name', 'n_sig', 'fs', 'counter_freq', 'base_counter', 'sig_len', 'base_time', 'base_date', 'comments', 'sig_name', 'p_signal', 'd_signal', 'e_p_signal', 'e_d_signal', 'file_name', 'fmt', 'samps_per_frame', 'skew', 'byte_offset', 'adc_gain', 'baseline', 'units', 'adc_res', 'adc_zero', 'init_value', 'checksum', 'block_size']
    record_dict = {k: v for k, v in example.items() if k in record_keys}
    record_dict['p_signal'] = np.array(record_dict['p_signal']) 
    record = wfdb.Record(**record_dict)

    wfdb.plot_wfdb(record, return_fig=True)
    record_name = record.__dict__['record_name']
    filename = f"/mnt/md0/helenjin/FIX-2/src/prompts/data/cardiac_relevance_fewshot_{i+1}_image_{annotation_examples[i]}.png"
    plt.savefig(filename, dpi=200, bbox_inches='tight')
    plt.close()
    
    # wfdb.plot_wfdb(record=record)

False None
False None
True 2206-05-24 14:29:37
True 2251-02-12 09:12:02
True 2271-01-08 12:14:22
